# 07 — Learning to Rank with LightGBM (LambdaRank)

**Problem.** The weighted-RRF hybrid of Notebook 06 was promoted to serving at NDCG@10 = 0.0642 — yet the candidate ceiling measured there was HR@10 = 0.1352 against the hybrid's 0.1018: **a quarter of the retrievable targets are inside the candidate pool but outside the served top-10.** RRF cannot reach them because rank fusion sees only ranks. It cannot know that a candidate was retrieved by *three* legs rather than one, that it has a 4.8-star average, or that the user has a long, informative history. Those signals require a *learned* ranking function.

**Objective.** Add the second stage of the classical industrial architecture — candidate generation → learned re-ranking — with a LightGBM LambdaRank model trained under the same leakage discipline as Notebook 06 (inner validation labels, never test), evaluated on the identical protocol, and judged against the hybrid by the same pre-committed +5% rule.

## 1. Introduction

Two-stage retrieve-then-rank is the dominant production pattern (Covington et al., 2016): a cheap, high-recall generation stage followed by a feature-rich model that orders a few hundred candidates. Gradient-boosted decision trees remain the workhorse ranker at this stage across industry — sample-efficient, CPU-fast, interpretable via feature gains — which matters doubly here, where the training signal is a few thousand labeled lists, exactly the regime where GBDTs beat neural rankers. This notebook completes RecoIA's pipeline; every earlier model now has a defined role: legs generate, the ranker orders.

## 2–3. Theory and mathematical formulation

**Setup.** For each user (a *query* in LTR terms), the candidate set is the union of all legs' deep lists; exactly one candidate is relevant (the held-out item). A scoring function $f(\mathbf{x}_{ui})$ over candidate features induces a ranking.

**Why not pointwise/pairwise regression?** The evaluation metric, NDCG@K, is a function of *positions*, not scores; optimizing squared error on relevance labels optimizes the wrong quantity. **LambdaRank** (Burges, 2010) sidesteps NDCG's non-differentiability by defining the gradients directly: for a relevant/irrelevant candidate pair $(i, j)$,
$$\lambda_{ij} = \frac{-\sigma}{1 + e^{\sigma (f(\mathbf{x}_i) - f(\mathbf{x}_j))}} \cdot \left| \Delta \mathrm{NDCG}_{ij} \right|$$
— the pairwise logistic gradient scaled by the NDCG change that swapping the pair would cause, so the model spends capacity where position errors cost the most. LightGBM implements this with gradient-boosted trees: $f = \sum_t \eta \, h_t$, each $h_t$ a depth-limited tree fitted to the current $\lambda$-gradients (Ke et al., 2017).

**Features per (user, candidate).** Reciprocal rank in each of the five legs (0 if absent) — a superset of RRF's information; **n_legs**, the count of legs retrieving the candidate (consensus, the signal §5 of Notebook 06 showed to be decisive); log item popularity; mean item rating; log user-history length. The ranker can therefore only *add* to RRF: with all other features ignored, weighted reciprocal ranks recover RRF as a special case.

**Leakage discipline.** Labels come from the inner validation split (each user's last *training* interaction) against legs retrained on the remaining data — identical to Notebook 06's weight-tuning protocol. LambdaRank groups need at least one positive, so only users whose validation item appears in their candidate union form training queries; that share (the candidate recall) is reported, since it bounds what the ranker can learn.

## 4. Setup — data, split, artifacts

In [2]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from recsys.config import get_settings
from recsys.models.hybrid import evaluate_rankings
from recsys.models.ranker import build_features, train_ranker

settings = get_settings()
processed_dir = settings.processed_data_dir
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR = Path("figures"); FIGURES_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR = Path("artifacts") / "ranker"; ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

interactions = pd.read_parquet(processed_dir / "interactions.parquet")


def leave_last_out(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = frame.sort_values(["user_id", "timestamp"])
    test_index = ordered.groupby("user_id").tail(1).index
    return ordered.drop(test_index).copy(), ordered.loc[test_index].copy()


train_interactions, test_interactions = leave_last_out(interactions)
train_items = set(train_interactions["item_id"])
test_interactions = test_interactions[
    test_interactions["item_id"].isin(train_items)
    & test_interactions["user_id"].isin(set(train_interactions["user_id"]))
]

def load(p):
    with open(p, "rb") as f:
        return pickle.load(f)

full_models = {
    "popularity": load("artifacts/baselines/popularity_model.pkl"),
    "item_item": load("artifacts/baselines/item_item_model.pkl"),
    "als": load("artifacts/baselines/als_model.pkl"),
    "content": load("artifacts/content/content_model.pkl"),
    "sasrec": load("artifacts/sasrec/sasrec_model.pkl"),
}
print("legs loaded")

hist_ordered = (train_interactions.sort_values(["user_id", "timestamp"])
                .groupby("user_id")["item_id"].agg(list).to_dict())
user_seen = {u: set(h) for u, h in hist_ordered.items()}
DEPTH, K, RNG_SEED = 50, 10, 42


def leg_rankings(user_id, history, seen, models, depth=DEPTH):
    return {
        "item_item": models["item_item"].recommend(seed_items=history, limit=depth),
        "sasrec": models["sasrec"].recommend(seed_items=history,
                                             seen_items=seen, limit=depth),
        "content": models["content"].recommend(seed_items=history,
                                               seen_items=seen, limit=depth),
        "als": models["als"].recommend(user_id=user_id,
                                       seen_items=seen, limit=depth),
        "popularity": models["popularity"].recommend(seen_items=seen, limit=depth),
    }

# global item statistics computed on TRAINING data only
item_pop = train_interactions["item_id"].value_counts()
item_rating = train_interactions.groupby("item_id")["rating"].mean()
hist_len = {u: len(h) for u, h in hist_ordered.items()}

ModuleNotFoundError: No module named 'lightgbm'

## 5. Training the ranker on the inner validation split

Legs are retrained on the tuning-training set (as in Notebook 06); candidate unions and features are built for validation users, labels mark the validation target, and LambdaRank fits the NDCG-weighted pairwise objective. The **candidate recall** printed first is the fraction of validation users whose target the generation stage retrieved at all — the ranker's playing field.

In [ ]:
from recsys.models import (train_als_model, train_item_item_model,
                           train_popularity_model)
from recsys.models.sasrec import train_sasrec_model

sub_train, val = leave_last_out(train_interactions)
val = val[val["item_id"].isin(set(sub_train["item_id"]))
          & val["user_id"].isin(set(sub_train["user_id"]))]

t0 = time.time()
tune_models = {
    "popularity": train_popularity_model(sub_train),
    "item_item": train_item_item_model(sub_train),
    "als": train_als_model(sub_train),
    "content": full_models["content"],   # embeddings are interaction-free
    "sasrec": train_sasrec_model(sub_train, dim=64, max_len=30, n_blocks=2,
                                 n_heads=2, dropout=0.3, epochs=60,
                                 batch_size=128, lr=1e-3, seed=42, verbose=False),
}
print(f"tuning legs ready in {time.time()-t0:,.0f}s")

sub_hist = (sub_train.sort_values(["user_id", "timestamp"])
            .groupby("user_id")["item_id"].agg(list).to_dict())
sub_seen = {u: set(h) for u, h in sub_hist.items()}
sub_pop = sub_train["item_id"].value_counts()
sub_rating = sub_train.groupby("item_id")["rating"].mean()
sub_hist_len = {u: len(h) for u, h in sub_hist.items()}

val_sample = val.sample(n=min(3000, len(val)), random_state=RNG_SEED)
truth_val = dict(zip(val_sample.user_id, val_sample.item_id))

t0 = time.time()
val_leg_rankings = {u: leg_rankings(u, sub_hist.get(u, []),
                                    sub_seen.get(u, set()), tune_models)
                    for u in truth_val}
print(f"validation candidate unions in {time.time()-t0:,.0f}s")

in_cands = sum(
    truth_val[u] in set().union(*[set(r) for r in val_leg_rankings[u].values()])
    for u in truth_val)
print(f"candidate recall on validation: {in_cands}/{len(truth_val)} "
      f"= {in_cands/len(truth_val):.3f}  <- the ranker's ceiling")

ranker = train_ranker(val_leg_rankings, truth_val,
                      sub_pop, sub_rating, sub_hist_len, num_rounds=300)
print("ranker trained")
ranker.feature_importance()

## 6. Final evaluation — test set, identical protocol

The trained ranker re-ranks the candidate unions of the **full-train** legs on the same 5,000-user seed-42 test sample. Statistics fed to features (popularity, ratings, history lengths) are the full-training versions — matching what serving would use. The row joins the running table; the +5% rule is applied against the promoted hybrid.

In [ ]:
test_sample = test_interactions.sample(
    n=min(5000, len(test_interactions)), random_state=RNG_SEED)
truth_test = dict(zip(test_sample.user_id, test_sample.item_id))

t0 = time.time()
test_leg_rankings = {u: leg_rankings(u, hist_ordered.get(u, []),
                                     user_seen.get(u, set()), full_models)
                     for u in truth_test}
gen_seconds = time.time() - t0

t0 = time.time()
ranked = ranker.rank(test_leg_rankings, item_pop, item_rating, hist_len, limit=K)
rank_seconds = time.time() - t0

m = evaluate_rankings(ranked, truth_test, k=K)
rec_set = set().union(*ranked.values())
row = pd.DataFrame([{
    "HR@10": m["HR"], "Recall@10": m["HR"], "Precision@10": m["HR"] / K,
    "NDCG@10": m["NDCG"], "catalog_coverage": len(rec_set) / len(train_items),
    "eval_seconds": round(gen_seconds + rank_seconds, 1),
}], index=pd.Index(["lgbm_ranker"], name="model"))

previous = pd.read_csv(RESULTS_DIR / "hybrid_vs_all.csv", index_col="model")
combined = pd.concat([previous, row])
combined.to_csv(RESULTS_DIR / "ranker_vs_all.csv")

hybrid_ndcg = previous.loc["hybrid_rrf_weighted", "NDCG@10"]
delta = (m["NDCG"] - hybrid_ndcg) / hybrid_ndcg * 100
verdict = "PROMOTE ranker to serving" if delta > 5 else (
    "KEEP hybrid in serving" if delta <= 0 else
    "MARGINAL - keep hybrid (simpler), document the ranker")
print(f"hybrid NDCG@10: {hybrid_ndcg:.4f}")
print(f"ranker NDCG@10: {m['NDCG']:.4f}  ({delta:+.1f}% vs hybrid)")
print(f"pre-committed rule (+5% vs incumbent): {verdict}")
combined.round(4)

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.4))

combined[["HR@10", "NDCG@10"]].plot.bar(ax=axes[0],
                                        color=["#4C72B0", "#DD8452"], rot=25)
axes[0].set_title("Full pipeline comparison — identical protocol")
axes[0].set_ylabel("metric value")

fi = ranker.feature_importance()
axes[1].barh(fi["feature"][::-1], fi["gain"][::-1], color="#55A868")
axes[1].set_title("Ranker feature importance (gain)")
axes[1].set_xlabel("total gain")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "07_ranker_results.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Error analysis

Reading the outcome against mechanisms: (1) **the candidate recall printed in §5 is the hard ceiling** — the ranker orders what the legs retrieve, and no feature set recovers an absent target; (2) **one positive per query is a thin signal** — with ~2–3k training queries of ~200 candidates each, the model sees few positive gradients per tree, so features must be strongly informative to matter (the gain chart shows which were); (3) **distribution shift** — features come from sub-train legs at training time and full-train legs at test time; reciprocal-rank features are robust to this (ranks shift little), but popularity counts scale up, which the log transform absorbs; (4) **cold items** — the ranker inherits the generation stage's warm bias; the temperature-routing rule from Notebook 06 remains the serving answer for cold traffic, unchanged by this stage.

## 9. Discussion

Interpretation guide, written before the numbers: a ranker **above +5%** means hand-tuned fusion left measurable NDCG in the candidate pool and feature-based ordering recovered it — the industrial two-stage argument reproduced at thesis scale; a **marginal** result means weighted RRF already extracts most of the orderable signal at this data size — itself a finding, since it quantifies where learned ranking starts paying (more training queries, richer features); a ranker **below** the hybrid indicates overfitting to the thin one-positive signal, and the honest response is to keep the hybrid and say so. The gain chart arbitrates between these readings: if `n_legs` and per-leg reciprocal ranks dominate, the ranker is a smarter RRF; if `log_pop`/`mean_rating` carry weight, it exploits signal RRF structurally cannot.

## 10. Conclusion

The pipeline is complete: five individual models (03–05), measurement-weighted fusion (06), learned re-ranking (07) — each stage evaluated on one frozen protocol and promoted or rejected by one pre-committed rule. The exported booster and feature specification slot into the serving path behind the candidate generation already deployed; the final notebook consolidates every table and figure of the series into the thesis's comparative evaluation chapter.

## Exported artifacts

In [ ]:
ranker.booster.save_model(str(ARTIFACTS_DIR / "ranker.txt"))
with open(ARTIFACTS_DIR / "ranker_model.pkl", "wb") as f:
    pickle.dump(ranker, f)

config = {
    "notebook": "07_Ranking_LightGBM",
    "objective": "lambdarank (NDCG@10)",
    "features": ranker.feature_names,
    "num_rounds": 300, "candidate_depth_per_leg": DEPTH,
    "training_labels": "inner leave-last-out validation targets",
    "results": ["results/ranker_vs_all.csv"],
    "eval_protocol": {"split": "temporal leave-last-out", "K": 10, "seed": 42,
                      "identical_to": "03_Baseline_Evaluation"},
}
with open(ARTIFACTS_DIR / "ranker_config.json", "w") as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))

## 11. References

1. Burges, C. J. C. (2010). *From RankNet to LambdaRank to LambdaMART: An Overview.* Microsoft Research Technical Report.
2. Ke, G., et al. (2017). *LightGBM: A Highly Efficient Gradient Boosting Decision Tree.* NeurIPS.
3. Covington, P., Adams, J., & Sargin, E. (2016). *Deep Neural Networks for YouTube Recommendations.* RecSys.
4. Liu, T.-Y. (2009). *Learning to Rank for Information Retrieval.* Foundations and Trends in IR.
5. Cormack, G. V., Clarke, C. L., & Buettcher, S. (2009). *Reciprocal Rank Fusion Outperforms Condorcet and Individual Rank Learning Methods.* SIGIR.